# Turning Comparisons Into a Reward Model

**Topic:** Bradley-Terry preference modelling &nbsp;·&nbsp; **Difficulty:** Medium

Nobody can score a model response on an absolute scale, but everybody can say
which of two is better. Bradley-Terry turns those pairwise judgements into a
scalar reward: `P(a beats b) = sigmoid(r_a - r_b)`, fitted by maximum likelihood.

The subtlety worth internalizing is that the fit is only identified up to an
additive constant: no amount of comparison data can tell you the absolute
level.

## The data (created in the next cell)

- 30 responses with true latent rewards.
- 2,897 noisy pairwise comparisons sampled from the Bradley-Terry model
  (3,000 candidate pairs, self-pairs discarded).
- `np.random.default_rng(5003)` draws the latent rewards and
  `np.random.default_rng(800)` draws the comparisons.

## Your task

With **NumPy only**:

1. Fit `r` `(30,)` by gradient ascent on the log-likelihood
   `sum log sigmoid(r_win - r_lose)`, 3000 steps at lr 0.5 applied to the
   mean gradient (`r += 0.5 * grad / n_pairs`), starting from zeros. Some items
   win many times, so accumulate repeated indices with `np.add.at` or
   `np.bincount`: plain `grad[win] += g` keeps only the last write at each index
   and the fit goes nowhere.
2. Because the model is shift-invariant, centre both the fitted and the true
   rewards before comparing.
3. Set `answers` to a dict with `"final_nll"` (mean negative log-likelihood),
   `"corr"` (Pearson correlation between centred fitted and centred true
   rewards), `"rank_agreement"` (fraction of held-out pairs whose predicted
   winner matches the true higher-reward item, using 4,000 candidate pairs
   from `np.random.default_rng(999)` and keeping the pairs with unequal
   items), `"shift_invariance"`: the
   change in mean NLL when 100 is added to every fitted reward (should be 0), and
   `"mean_fitted"` (the mean of the fitted rewards before centring).

## Evaluation

Compared to the reference to a relative tolerance of 1e-5. The evaluation cell
checks the fit correlates strongly with the truth and that shifting every reward
leaves the likelihood unchanged.

In [ ]:
import numpy as np

rng = np.random.default_rng(5003)
n_items = 30
true_r = rng.normal(0, 1.2, n_items)

rs = np.random.default_rng(800)
n_comp = 3000
ii = rs.integers(0, n_items, n_comp)
jj = rs.integers(0, n_items, n_comp)
keep = ii != jj
ii, jj = ii[keep], jj[keep]
pwin = 1 / (1 + np.exp(-(true_r[ii] - true_r[jj])))
i_wins = rs.random(len(ii)) < pwin
win = np.where(i_wins, ii, jj)
lose = np.where(i_wins, jj, ii)
print(f"{n_items} responses, {len(win)} pairwise comparisons")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("final_nll", [], 0.5209572013766502, -0.08722803960182358, 1e-05),
    ("corr", [], 0.9815187627191315, -0.16434355313287483, 1e-05),
    ("rank_agreement", [], 0.8988033298647242, -0.15049384526121087, 1e-05),
    ("shift_invariance", [], 0.0, 0.0, 1e-05),
    ("mean_fitted", [], 7.401486830834377e-17, -1.2392902616305668e-17, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "corr" in answers:
    _fit = answers["corr"] > 0.9 and answers["rank_agreement"] > 0.8
    _shift = answers["shift_invariance"] < 1e-9
    print(f"  {'recovers latent reward':<24} {'ok' if _fit else 'wrong'}  "
          f"(corr {answers['corr']:.3f}, rank agreement {answers['rank_agreement']:.3f})")
    print(f"  {'likelihood shift-invariant':<24} {'ok' if _shift else 'wrong'}")
    _ok = _ok and _fit and _shift

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Every term in the likelihood depends only on `r_win - r_lose`, so adding a
constant to every reward changes nothing. The model is identified only up to that
shift, which is why you always centre before comparing or reporting. The fitted
mean is also zero here rather than arbitrary: the updates start from zeros and
every gradient sums to zero, so the fit stays centred. In RLHF this is harmless
for ranking but matters the moment the reward is used in a KL-regularized
objective, where the absolute scale trades against the KL coefficient.

```python
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

r = np.zeros(n_items)
n = len(win)
for _ in range(3000):
    d = r[win] - r[lose]
    g = sigmoid(-d)
    grad = np.zeros(n_items)
    np.add.at(grad, win, g)
    np.add.at(grad, lose, -g)
    r += 0.5 * grad / n

d = r[win] - r[lose]
final_nll = float(-np.log(sigmoid(d) + 1e-12).mean())

rc = r - r.mean()
tc = true_r - true_r.mean()
corr = float(np.corrcoef(rc, tc)[0, 1])

rs2 = np.random.default_rng(999)
a = rs2.integers(0, n_items, 4000)
b = rs2.integers(0, n_items, 4000)
m = a != b
a, b = a[m], b[m]
agree = float(((r[a] > r[b]) == (true_r[a] > true_r[b])).mean())

r_shift = r + 100.0
d2 = r_shift[win] - r_shift[lose]
nll2 = float(-np.log(sigmoid(d2) + 1e-12).mean())

answers = {
    "final_nll": final_nll,
    "corr": corr,
    "rank_agreement": agree,
    "shift_invariance": float(abs(nll2 - final_nll)),
    "mean_fitted": float(r.mean()),
}
```